# CodeGen — Group 45

## Step 8 — the large-model reference: Qwen2.5-Coder-7B in 4-bit

**Why this row.** Every inference-time lever we have on the 1.5B is now spent. The
compile-guided cascade (Step 6) reached 44.9% and compiler-feedback repair (Step 7/7b)
pushed the best inference-legal system to 46.2%. The error analysis says the wall is
not compilation any more — it is **wrong logic**: ~77 of the 156 problems compile and
run but return the wrong answer, and a 1.5B simply does not reason well enough to fix
them. Compiler feedback cannot touch a program that already compiles.

The proposal also asks for a **large-LLM comparison column**. Both needs are answered
by the same experiment: run **Qwen2.5-Coder-7B** — the 7-billion-parameter sibling of
our subject model, so it is a clean scale comparison — on the *identical* harness and
prompts, with no RAG and no repair. That isolates one variable: model size.

**How it fits a T4.** 7B in fp16 is ~15 GB and will not fit a 16 GB T4 alongside
activations, so we load it in **4-bit** (bitsandbytes nf4, fp16 compute, double
quant) — ~5 GB of weights, comfortable headroom for 512-token greedy generation.
4-bit is the standard way to run a 7B on a T4; it costs a little quality but keeps the
comparison honest and reproducible on free Colab.

| Same harness, same 156 problems | Score |
|---|---|
| Qwen-1.5B vanilla (Step 5) | 37.8% |
| Qwen-1.5B compile-guided cascade (Step 6) | 44.9% |
| Qwen-1.5B cascade + compiler-feedback repair (Step 7b) | 46.2% |
| **Qwen-7B-4bit vanilla (this step)** | **measured below** |
| oracle union across 1.5B configs | 54.5% |

House rules apply. Sections 0-5 are the Step 5/6 harness unchanged (so this row is
directly comparable); the 4-bit 7B load and the plain sweep are the only new pieces.

## 0. Colab setup — Drive + Hugging Face token (run this first)

Everything we produce (benchmark file, model copy, eval results) lives in Drive at
`MyDrive/CodeGen_Group45`, so a crashed or recycled Colab session never loses work.

**One-time setup:** add a Colab secret (key icon in the left sidebar) named `HF_TOKEN`
containing a Hugging Face **read** token, and switch **Notebook access** ON for it.
Unauthenticated downloads from Colab are exactly what stalls / 403s (July 2026).

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/CodeGen_Group45"
    for sub in ("data", "models", "eval"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

    # HF auth BEFORE anything talks to the Hub. Colab secret: HF_TOKEN, Notebook access ON.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secret")
    except Exception as e:
        print(f"WARNING: could not read the HF_TOKEN secret ({type(e).__name__}). "
              "Hub downloads may stall or 403 — add the secret and enable Notebook access.")
else:
    print("Not on Colab — skipping Drive; the benchmark loads from the repo's data/ folder.")

# Escape hatch only — leave False. With an upgraded hf_xet + auth, the Xet backend is the
# path that works from Colab; the non-Xet fallback was 403ing server-side (July 2026).
DISABLE_XET = False
if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

print("DRIVE_ROOT =", DRIVE_ROOT)

Mounted at /content/drive
HF token loaded from Colab secret
DRIVE_ROOT = /content/drive/MyDrive/CodeGen_Group45


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [2]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.97.1 (8bab26

## 2. Install Python dependencies

Only `huggingface_hub` + its `hf_xet` download backend — and we **upgrade** them, because
Colab's preinstalled `hf_xet` is exactly what stalled our model downloads.

**Deliberately NOT installed: `datasets`.** `pip install -U datasets` drags a newer pyarrow
over Colab's preinstalled one and crashes the runtime (`IpcReadOptions size changed`).
This notebook never imports `datasets` at all — the benchmark is a plain JSONL (Section 3).

In [3]:
# Upgrade the Hub client + Xet backend BEFORE anything imports huggingface_hub.
# Do NOT add `datasets` or `torch` here (see the markdown above).
!pip install -q -U huggingface_hub hf_xet
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 111.0 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

We keep the benchmark as a plain JSONL file (repo: `data/humaneval_rs.jsonl`, Drive:
`CodeGen_Group45/data/humaneval_rs.jsonl`) and read it with stdlib `json` — no `datasets`
library, no Hub download, nothing to flake. `ds` is a plain list of dicts.

In [4]:
import json, os

def load_benchmark():
    candidates = []
    if DRIVE_ROOT:
        candidates.append(os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl"))
    candidates += ["data/humaneval_rs.jsonl", "../data/humaneval_rs.jsonl"]  # repo checkout
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                problems = [json.loads(line) for line in f if line.strip()]
            print(f"Loaded {len(problems)} problems from cache: {path}")
            return problems

    # Last resort (no Hub involved): hand-upload the repo's data/humaneval_rs.jsonl,
    # then stash it on Drive so this never happens again.
    if IN_COLAB:
        from google.colab import files
        print("Benchmark not found on Drive. Upload data/humaneval_rs.jsonl from the repo:")
        uploaded = files.upload()
        raw = next(iter(uploaded.values()))
        problems = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
        if DRIVE_ROOT:
            dest = os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl")
            with open(dest, "wb") as f:
                f.write(raw)
            print("Cached to Drive:", dest)
        return problems
    raise FileNotFoundError("humaneval_rs.jsonl not found — expected in the repo's data/ "
                            "folder or on Drive under CodeGen_Group45/data/.")

ds = load_benchmark()
assert len(ds) == 156, f"expected 156 problems, got {len(ds)}"
assert all(k in ds[0] for k in ("name", "prompt", "tests", "stop_tokens"))

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])

Loaded 156 problems from cache: /content/drive/MyDrive/CodeGen_Group45/data/humaneval_rs.jsonl

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [5]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")

harness ready


## 5. We self-test the harness (most important step)

---


Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [6]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\n Harness works correctly — it can tell good Rust from bad.")

correct -> pass
wrong   -> run_fail
broken  -> compile_error

 Harness works correctly — it can tell good Rust from bad.


## 6a. Dependencies for 4-bit loading

On top of the Step 5/6 stack we need **bitsandbytes** (the nf4 kernels) and a recent
**accelerate** (for `device_map`). Both are wheels-only installs — no `datasets`, so the
Colab pyarrow trap from Section 2 stays clear.

In [7]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes", "accelerate", "transformers"], check=True)
import bitsandbytes as bnb
print("bitsandbytes", bnb.__version__, "ready")

bitsandbytes 0.50.0 ready


## 6b. Fetch the 7B weights

Same stall-proof acquisition ladder as Steps 5/6, pointed at the 7B checkpoint:
**Drive copy** (trusted only with the `_SAVED_OK` marker) -> **ModelScope** (primary —
HF-from-Colab stalled in July 2026) -> **HF Hub** (last resort, killable subprocess).

Two things differ from the 1.5B:

- The raw download is ~15 GB and lands on Colab's **local disk** (`/content`, ~80 GB
  free), never on Drive — free Drive is only 15 GB and could not hold it.
- What we *cache* on Drive is the **4-bit** model (~5 GB, produced after the first
  quantized load in Section 6c), so later sessions skip the 15 GB download. If Drive is
  tight, set `SAVE_7B_TO_DRIVE = False` below and the notebook simply re-downloads each
  session (results still stream/resume to Drive regardless).

In [8]:
import os, shutil, subprocess, sys, json as _json

MODEL_ID  = "Qwen/Qwen2.5-Coder-7B"      # base (completion) sibling of the 1.5B subject model
MARKER    = "_SAVED_OK"
DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "qwen25coder-7b-4bit") if DRIVE_ROOT else None
LOCAL_4BIT_DIR  = "/content/qwen25coder-7b-4bit"     # local copy of the Drive 4-bit cache
SAVE_7B_TO_DRIVE = True                              # flip to False if free Drive is full

def _modelscope_download():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modelscope"], check=True)
    from modelscope import snapshot_download
    return snapshot_download(MODEL_ID)

def _hf_download():
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}')"
    for attempt in (1, 2):
        try:
            subprocess.run([sys.executable, "-c", code], check=True, timeout=1800)  # 7B is big
            from huggingface_hub import snapshot_download
            return snapshot_download(MODEL_ID, local_files_only=True)
        except subprocess.TimeoutExpired:
            print(f"HF Hub attempt {attempt}: no finish within 30 min (stalled) — killed")
        except subprocess.CalledProcessError:
            print(f"HF Hub attempt {attempt}: download process errored")
    raise RuntimeError(
        "All hubs failed for " + MODEL_ID + " (Drive empty, ModelScope failed, HF stalled). "
        "Download it on another machine and upload the 4-bit copy to Drive under "
        "models/qwen25coder-7b-4bit with an empty _SAVED_OK file.")

def fetch_model_dir():
    """Return (dir, prequantized). Order: Drive 4-bit cache -> ModelScope -> HF Hub."""
    # (1) Drive 4-bit cache — small, already quantized; copy to local disk before loading.
    if DRIVE_MODEL_DIR and os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
        if not os.path.exists(os.path.join(LOCAL_4BIT_DIR, MARKER)):
            print("4-bit 7B found on Drive — copying to local disk (one-time per session)...")
            shutil.copytree(DRIVE_MODEL_DIR, LOCAL_4BIT_DIR, dirs_exist_ok=True)
        print("Using the Drive 4-bit copy")
        return LOCAL_4BIT_DIR, True
    # (2) ModelScope — primary hub. Raw ~15 GB to local disk.
    try:
        path = _modelscope_download()
        print("Downloaded raw 7B from ModelScope")
        return path, False
    except Exception as e:
        print(f"ModelScope failed: {type(e).__name__}: {e}")
    # (3) HF Hub — last resort, stall-proofed.
    path = _hf_download()
    print("Downloaded raw 7B from the Hugging Face Hub")
    return path, False

model_dir, prequantized = fetch_model_dir()
# be robust: a raw dir may still turn out to carry a quantization_config
cfgp = os.path.join(model_dir, "config.json")
if os.path.exists(cfgp):
    prequantized = prequantized or ("quantization_config" in _json.load(open(cfgp)))
print("model files at:", model_dir, "| already 4-bit:", prequantized)

2026-07-30 12:02:34,842 | INFO    | modelscope_hub.download | Downloading 15 files from Qwen/Qwen2.5-Coder-7B@master


Downloading:   0%|          | 0/15 [00:00<?, ?file/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

configuration.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Downloaded raw 7B from ModelScope
model files at: /root/.cache/modelscope/models/Qwen--Qwen2.5-Coder-7B/snapshots/master | already 4-bit: False


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU, then rerun."

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bf16
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(model_dir)
if prequantized:
    # config already carries the quantization_config; do not pass it again
    model = AutoModelForCausalLM.from_pretrained(model_dir, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_dir, quantization_config=BNB, device_map="auto")
model.eval()
GEN_DEVICE = "cuda"   # single-GPU device_map keeps everything on cuda:0
print("7B loaded 4-bit;", f"{model.get_memory_footprint()/1e9:.1f} GB on GPU")

# One-time: stash the 4-bit copy on Drive (~5 GB) so later sessions skip the 15 GB download.
if (not prequantized and SAVE_7B_TO_DRIVE and DRIVE_MODEL_DIR
        and not os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER))):
    try:
        print("Saving 4-bit copy to Drive (one-time, ~5 GB, a few minutes)...")
        model.save_pretrained(DRIVE_MODEL_DIR)
        tok.save_pretrained(DRIVE_MODEL_DIR)
        with open(os.path.join(DRIVE_MODEL_DIR, MARKER), "w") as f:
            f.write("ok\n")
        print("Saved to", DRIVE_MODEL_DIR)
    except Exception as e:
        print(f"Drive save skipped ({type(e).__name__}: {e}). "
              "Not fatal — the run continues; next session will re-download.")

def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text

print("trim_to_body ready")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

7B loaded 4-bit; 5.4 GB on GPU
Saving 4-bit copy to Drive (one-time, ~5 GB, a few minutes)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/CodeGen_Group45/models/qwen25coder-7b-4bit
trim_to_body ready


## 7. The vanilla 7B sweep (+ smoke test)

Exactly the Step 5 baseline recipe, only the model is bigger: feed each MultiPL-E
prompt as-is (no RAG, no repair), greedy decode, stop at MultiPL-E's `"\n}"`, and trim
to the function body. Same `evaluate_one`, so the resulting pass rate sits on the same
axis as every 1.5B number.

Results stream per-problem to Drive (`eval/step8_qwen7b_4bit.jsonl`) and resume on
re-run — important, because a 7B on a T4 is slower than the 1.5B, so a dropped session
must never mean redoing finished problems. Smoke test first (house rule): 5 problems on
a throwaway file, asserting they do not all fail to compile.

In [10]:
import time, json
from collections import Counter

EVAL_DIR = os.path.join(DRIVE_ROOT, "eval") if DRIVE_ROOT else "."

def qwen7b_completion(prompt_text, max_new_tokens=512):
    inputs = tok(prompt_text, return_tensors="pt").to(GEN_DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)  # MultiPL-E's stop token
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

def run_7b(limit=None, tag=""):
    path = os.path.join(EVAL_DIR, f"step8_qwen7b_4bit{tag}.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                done[rec["name"]] = rec["status"]
    data = ds if limit is None else ds[:limit]
    todo = [ex for ex in data if ex["name"] not in done]
    print(f"7B{tag}: {len(done)} already done, {len(todo)} to go -> {path}")
    t0 = time.time()
    with open(path, "a") as out:
        for ex in todo:
            body = qwen7b_completion(ex["prompt"])
            status = evaluate_one(ex["prompt"], body, ex["tests"])
            out.write(json.dumps({"name": ex["name"], "status": status, "body": body}) + "\n")
            out.flush()
            done[ex["name"]] = status
            print(f"[{len(done):3d}/{len(data)}] {ex['name'][:40]:40s} {status:14s} ({time.time()-t0:5.0f}s)")
    counts = Counter(done.values())
    print(f"7B{tag}: pass {100*counts['pass']/len(done):.1f}%  {dict(counts)}")
    return counts

# --- smoke test: 5 problems, throwaway file (house rule) ---
smoke = run_7b(limit=5, tag="_smoke")
os.remove(os.path.join(EVAL_DIR, "step8_qwen7b_4bit_smoke.jsonl"))
assert smoke["compile_error"] < 5, ("every smoke problem failed to compile — print one body "
                                    "before spending GPU time on all 156")
print("\nsmoke OK — the full 7B sweep below is safe to launch")

7B_smoke: 0 already done, 5 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step8_qwen7b_4bit_smoke.jsonl
[  1/5] HumanEval_0_has_close_elements           pass           (   16s)
[  2/5] HumanEval_1_separate_paren_groups        pass           (   41s)
[  3/5] HumanEval_2_truncate_number              pass           (   45s)
[  4/5] HumanEval_3_below_zero                   pass           (   57s)
[  5/5] HumanEval_4_mean_absolute_deviation      pass           (   66s)
7B_smoke: pass 100.0%  {'pass': 5}

smoke OK — the full 7B sweep below is safe to launch


## 8. The full 156-problem run

Greedy and deterministic, so it is safely re-runnable — if the T4 session drops, just
re-run this cell and it picks up where it stopped. Expect it to be noticeably slower
per problem than the 1.5B (4-bit dequant on every matmul), on the order of tens of
minutes for the full set.

In [11]:
counts7b = run_7b()
print(f"\nQwen-7B-4bit vanilla: pass {100*counts7b['pass']/len(ds):.1f}%  {dict(counts7b)}")

7B: 0 already done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step8_qwen7b_4bit.jsonl
[  1/156] HumanEval_0_has_close_elements           pass           (    8s)
[  2/156] HumanEval_1_separate_paren_groups        pass           (   19s)
[  3/156] HumanEval_2_truncate_number              pass           (   21s)
[  4/156] HumanEval_3_below_zero                   pass           (   26s)
[  5/156] HumanEval_4_mean_absolute_deviation      pass           (   32s)
[  6/156] HumanEval_5_intersperse                  pass           (   39s)
[  7/156] HumanEval_6_parse_nested_parens          pass           (   49s)
[  8/156] HumanEval_7_filter_by_substring          pass           (   52s)
[  9/156] HumanEval_8_sum_product                  pass           (   55s)
[ 10/156] HumanEval_9_rolling_max                  run_fail       (   60s)
[ 11/156] HumanEval_10_make_palindrome             run_fail       (   69s)
[ 12/156] HumanEval_11_string_xor                  pass           (   72s)

## 9. Results — does scale buy what inference tricks could not?

This row isolates one question: the 1.5B's remaining misses are wrong-logic `run_fail`s,
which no compiler-based method can repair. If a 7B — same family, same prompts — clears
a meaningful share of those, that is evidence the wall was capability, not our pipeline.
We print the vanilla 7B against every 1.5B configuration, then break down where the 7B
still fails.

In [12]:
p7 = counts7b["pass"]
print("Same harness, same 156 problems")
print(f"  Qwen-1.5B vanilla (Step 5)            :  59/156 = 37.8%")
print(f"  Qwen-1.5B cascade (Step 6)            :  70/156 = 44.9%")
print(f"  Qwen-1.5B cascade + repair (Step 7b)  :  72/156 = 46.2%")
print(f"  Qwen-7B-4bit vanilla (this step)      : {p7:3d}/156 = {100*p7/len(ds):.1f}%")
print()
print(f"7B failure breakdown: run_fail {counts7b['run_fail']} (wrong logic) | "
      f"compile_error {counts7b['compile_error']} | run_timeout {counts7b['run_timeout']}")
print("The interesting comparison is run_fail: the 1.5B baseline had 62 and no inference")
print("trick moved them. Whatever the 7B shaves off that count is pure capability gain.")

Same harness, same 156 problems
  Qwen-1.5B vanilla (Step 5)            :  59/156 = 37.8%
  Qwen-1.5B cascade (Step 6)            :  70/156 = 44.9%
  Qwen-1.5B cascade + repair (Step 7b)  :  72/156 = 46.2%
  Qwen-7B-4bit vanilla (this step)      :  91/156 = 58.3%

7B failure breakdown: run_fail 40 (wrong logic) | compile_error 23 | run_timeout 2
The interesting comparison is run_fail: the 1.5B baseline had 62 and no inference
trick moved them. Whatever the 7B shaves off that count is pure capability gain.


## 10. (Optional) Best-of small+large — the combined oracle

Cheap, high-value for the report: if the Step 6/7b files are on Drive, we can ask what a
*selector* that picks between the 1.5B cascade and the vanilla 7B could reach — the union
of problems either one solves. It is an upper bound (a real selector would need a chooser),
but it quantifies how complementary the small-pipeline and the large-model actually are.

In [13]:
def load_status_map(fname):
    path = os.path.join(EVAL_DIR, fname)
    if not os.path.exists(path):
        return None
    d = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            d[r["name"]] = r["status"]
    return d if len(d) == len(ds) else None

# rebuild the Step 6 cascade status per problem (first body that compiles)
k0 = load_status_map("step6_qwen_rag_k0.jsonl")
idi = load_status_map("step6_qwen_rag_k2_idiom.jsonl")
k4 = load_status_map("step6_qwen_rag_k4.jsonl")
seven = load_status_map("step8_qwen7b_4bit.jsonl")

if k0 and idi and k4 and seven:
    def casc_status(name):
        for src in (k0, idi, k4):
            if src[name] != "compile_error":
                return src[name]
        return k4[name]
    casc_pass = {ex["name"] for ex in ds if casc_status(ex["name"]) == "pass"}
    seven_pass = {ex["name"] for ex in ds if seven[ex["name"]] == "pass"}
    union = casc_pass | seven_pass
    print(f"1.5B cascade solves : {len(casc_pass)}/156 = {100*len(casc_pass)/len(ds):.1f}%")
    print(f"7B vanilla solves   : {len(seven_pass)}/156 = {100*len(seven_pass)/len(ds):.1f}%")
    print(f"either one solves   : {len(union)}/156 = {100*len(union)/len(ds):.1f}%  (best-of oracle)")
    print(f"only the 7B solves  : {len(seven_pass - casc_pass)}  "
          f"(problems the whole 1.5B pipeline never reached)")
    print(f"only the cascade    : {len(casc_pass - seven_pass)}  "
          f"(where retrieval+repair still beats raw scale)")
else:
    print("Need step6_qwen_rag_k0/k2_idiom/k4 and step8_qwen7b_4bit on Drive for the oracle.")
    print("Run Step 6 and this notebook's Section 8 first.")

1.5B cascade solves : 70/156 = 44.9%
7B vanilla solves   : 91/156 = 58.3%
either one solves   : 102/156 = 65.4%  (best-of oracle)
only the 7B solves  : 32  (problems the whole 1.5B pipeline never reached)
only the cascade    : 11  (where retrieval+repair still beats raw scale)


## What this step adds

- The **large-LLM reference column** the proposal asks for, on the exact same harness and
  prompts as every 1.5B row — a clean, single-variable scale comparison.
- Direct evidence on the central claim of the last three steps: the 1.5B's residual
  misses are wrong-logic `run_fail`s that inference-time methods cannot repair. The 7B's
  `run_fail` count (Section 9) shows how much of that wall is pure capability.
- A best-of-small+large oracle (Section 10) that quantifies how complementary retrieval
  plus repair on a small model is with raw scale — useful framing for the report.

With this row measured, the remaining work is assembly, not experiment: fold every number
into the CP3 comparison table (codegen-350M "before" -> Qwen-1.5B vanilla/RAG/cascade/repair
-> Qwen-7B-4bit large-model), and finalize the demo for deployment.